# Solve Field

This notebook can be used to determine the WCS solution of an OmegaCAM image.
It uses astrometry.net solve_field command. It is a necessary step before running scamp which needs a WCS accurate to approx 1", otherwise it fails.

Before start please be sure you can run the solve_field command from the astrometry package by tipying
/usr/local/astrometry/bin/solve-field

If you get something like
ERROR: You didn't specify any files to process.<br>
This program is part of the Astrometry.net suite.<br>
For details, visit http://astrometry.net.<br>
Git URL https://github.com/dstndstn/astrometry.net<br>
Revision 0.89, date Mon_Jan_17_07:47:26_2022_-0500.<br>

then you are okay.

This notebook will create a independent fits image for each CCD within the input mosaic. Then run solve_field on each one to find an astrometric solution. This solution will be incorporated to a new mosaic which will be use later.
Please be sure you have enough space in your hard disk, as this notebook will effectively created two new mosaic images for each input one.

Tested on my personal laptop using the index fields 4100 and 5200 series from astromatic.
Known issues:
- For some reason, the astrometry package failed during the flow of the script, but running the same command in terminal directly over the CCD fits image retrieves a solution in seconds.
- For those CCDs that have failed at retrieve a solution with astrometry, are copied directly into the final mosaic, causing some shifts between CCDs.

Lientur

In [ ]:
import os
import sys

import glob
import logging
import multiprocessing

sys.path.append(os.path.join(os.path.abspath(''), '..')) # to be removed in production
from pycavi import pycavi_solve
from pycavi import pycavi_tools
from pycavi import pycavi_instrument

In [ ]:
import multiprocessing

multiprocessing.cpu_count()

In [ ]:
# Set the reduced science dir where it will look for the reduced science images, same as previous notebooks
# For each target within the folder will check the filters before start to work on each observation.
# In this case, during the tests I consider only the images in the filter NB_659.

config_dir = os.getcwd()+'/'+'../config/'

config = pycavi_tools.read_pycavi_config(config_dir)
reduced_science_dir = config['Directories']['reduced_science_dir']
reduced_std_dir = config['Directories']['reduced_std_dir']
log_dir = config['Directories']['project_logs_dir']

# Should read the instrument configuration file. This ensure
# proper keyword parsing.
#
instrument = pycavi_instrument.Instrument(config_dir=config_dir, instrument='ocam')

# a list with the tags of image filters. During the tests I worked only with r_SDSS and NB_659
filters_to_sol = ["BB_V_89_ESO843", "BB_B_99_ESO842", "g_SDSS", "r_SDSS", "NB_659"]

# target list will include all the observed targets within the reduced_science directory
target_list = []
for object_ in glob.glob(reduced_science_dir + "*"):
    target_list.append(object_)

for object_ in glob.glob(reduced_std_dir + "*"):
    target_list.append(object_)
    
target_list.sort()

# for each observed target will all the images which filters correspond to the ones given in filters_to_sol
for target_ in target_list:
    for filter_ in filters_to_sol:
        for observation_ in glob.glob(target_ + "/" + filter_ + "/*"):
                print(observation_)
                pycavi_solve.run_py4_solve(observation_, instrument, cpu_count=multiprocessing.cpu_count()-2, redo_astrometry=False)

# job done
logging.info("\n\n ****** Py4 Done! ******\n\n")